# UPUR (Uang Palsu & Uang Rusak) – KPw BI Pematang Siantar 2018–2025
Notebook ini melakukan **parsing data pivot table mentah** dari file `uipur_2019-2025.xlsx`,
membersihkannya menjadi data tidy, lalu melakukan EDA singkat.

Output akhir: `yearly_totals.csv` dan `palsu_by_tahun_emisi.csv` yang akan dipakai oleh `app.py` (Streamlit).

**Cara pakai di Google Colab:**
1. Upload file `uipur_2019-2025.xlsx` ke Colab (atau mount Google Drive)
2. Jalankan semua cell secara berurutan
3. Download `data/yearly_totals.csv` dan `data/palsu_by_tahun_emisi.csv`, lalu commit ke folder `data/` di repo GitHub kamu


In [ ]:
# 1. Install & import library
!pip install -q openpyxl pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')


In [ ]:
# 2. Upload file Excel (jalankan cell ini jika belum ada file di Colab)
from google.colab import files
uploaded = files.upload()  # pilih file uipur_2019-2025.xlsx
FILE_PATH = list(uploaded.keys())[0]


In [ ]:
# Jika kamu pakai Google Drive, comment cell di atas dan pakai ini sebagai gantinya:
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/path/ke/uipur_2019-2025.xlsx'


## 3. Memahami struktur data
File ini adalah hasil *export* Pivot Table Excel. Setiap tahun (2018–2025) punya **blok terpisah**
yang disusun vertikal dalam satu sheet, dan strukturnya **tidak konsisten** antar tahun:
- Tahun 2018–2019: hanya kategori `Palsu` & `Palsu - Menunggu Klasifikasi`
- Tahun 2020 ke atas: ada breakdown per **tahun emisi** uang (2001, 2004, 2005, 2014, 2016, 2022, dst), dan kategori tambahan seperti `Asli`, `Asli - Rusak`

Karena itu kita perlu parsing manual per blok tahun, bukan `pd.read_excel` biasa.


In [ ]:
# 4. Baca raw sheet tanpa header
df = pd.read_excel(FILE_PATH, header=None)
print('Ukuran data:', df.shape)
df.head(10)


In [ ]:
# 5. Cari baris awal tiap blok tahun (baris yang kolom pertamanya berisi angka tahun, mis. 2018, 2019, ...)
# Catatan: baris label sub-kategori (tahun emisi uang) di dalam blok juga bisa berupa angka tahun,
# jadi kita validasi dengan mengecek baris berikutnya harus berupa header pivot table.

candidate_years = []
for i, v in enumerate(df[0]):
    if isinstance(v, (int, float)) and not pd.isna(v) and 2015 < v < 2030:
        next_row = df.iloc[i+1].tolist()[:2]
        text = str(next_row[0]) + str(next_row[1])
        if 'Jumlah' in text or 'Hitung' in text or 'Label' in text or 'Column' in text or (i+1 < len(df) and pd.isna(df.iloc[i+1,0])):
            candidate_years.append(i)

print('Baris awal blok tahun ditemukan di index:', candidate_years)


In [ ]:
# 6. Tentukan batas blok (start & end index) -- hasil validasi manual dari struktur file ini
starts = candidate_years
ends = starts[1:] + [len(df)]

for s in starts:
    print(f"Blok tahun {int(df.iloc[s,0])} mulai di baris {s}")


In [ ]:
# 7. Parsing tiap blok: ambil header kategori (Asli/Palsu/dll), sub-label (tahun emisi),
#    dan baris Grand Total / Total Keseluruhan di akhir blok

records = []

for s, e in zip(starts, ends):
    year = int(df.iloc[s, 0])
    block = df.iloc[s:e].reset_index(drop=True)

    # cari baris header ke-2 ('Row Labels' / 'Label Baris')
    hdr2_idx = None
    for i in range(min(8, len(block))):
        if str(block.iloc[i, 0]).strip() in ('Row Labels', 'Label Baris'):
            hdr2_idx = i
            break
    if hdr2_idx is None:
        continue

    hdr1 = block.iloc[hdr2_idx - 1].tolist()   # baris kategori utama (Asli/Palsu/dst)
    hdr2 = block.iloc[hdr2_idx].tolist()       # baris sub-label (tahun emisi)

    # cari baris Grand Total / Total Keseluruhan (dari bawah ke atas)
    gt_idx = None
    for i in range(len(block) - 1, -1, -1):
        if str(block.iloc[i, 0]).strip() in ('Grand Total', 'Total Keseluruhan'):
            gt_idx = i
            break
    if gt_idx is None:
        continue
    gt_row = block.iloc[gt_idx].tolist()

    # forward-fill kategori utama ke kanan (karena di Excel sel kategori sering merge/kosong)
    main_cat = [None] * len(hdr1)
    cur = None
    for c in range(1, len(hdr1)):
        v = hdr1[c]
        if isinstance(v, str) and v.strip():
            cur = v.strip()
        main_cat[c] = cur

    for c in range(1, len(hdr1)):
        val = gt_row[c]
        if pd.isna(val) or main_cat[c] is None:
            continue
        sub = hdr2[c]
        sub_label = None if (isinstance(sub, float) and pd.isna(sub)) else str(sub).strip()
        records.append({
            'Year': year,
            'MainCategory': main_cat[c],
            'SubLabel': sub_label,
            'Count': float(val)
        })

raw_parsed = pd.DataFrame(records)
raw_parsed.head(20)


In [ ]:
# 8. Normalisasi kategori untuk tahun 2018 & 2019 (struktur header sedikit beda: 'Label Kolom' / 'Column Labels')
def fix_cat(row):
    if row['MainCategory'] in ('Label Kolom', 'Column Labels'):
        return 'Palsu'
    return row['MainCategory']

raw_parsed['MainCategory'] = raw_parsed.apply(
    lambda r: 'Palsu' if r['MainCategory'] in ('Label Kolom', 'Column Labels') and 'Palsu' in str(r['SubLabel']) else r['MainCategory'],
    axis=1
)
raw_parsed.to_csv('data/uipur_raw_parsed.csv', index=False)
raw_parsed.tail(20)


In [ ]:
# 9. Bangun tabel ringkasan tahunan: Asli, Asli Rusak, Palsu, Palsu Menunggu Klasifikasi, Grand Total
import os
os.makedirs('data', exist_ok=True)

years = sorted(raw_parsed['Year'].unique())
rows = []
for y in years:
    sub = raw_parsed[raw_parsed['Year'] == y]

    def get(cat):
        m = sub[sub['MainCategory'] == cat]
        return m['Count'].sum() if len(m) else 0

    asli = get('Asli')
    asli_rusak = get('Asli - Rusak')
    palsu_menunggu = sub[sub['SubLabel'] == 'Palsu - Menunggu Klasifikasi']['Count'].sum()

    grand = sub[sub['MainCategory'].isin(['Grand Total', 'Total Keseluruhan'])]['Count'].sum()
    if grand == 0:
        grand = sub[sub['SubLabel'].isin(['Grand Total', 'Total Keseluruhan'])]['Count'].sum()
        palsu = sub[sub['SubLabel'] == 'Palsu']['Count'].sum()
    else:
        palsu = get('Palsu Total')
        if palsu == 0:
            palsu = sub[sub['MainCategory'] == 'Palsu']['Count'].sum()

    rows.append({
        'Year': y,
        'Asli': asli,
        'Asli_Rusak': asli_rusak,
        'Palsu': palsu,
        'Palsu_Menunggu_Klasifikasi': palsu_menunggu,
        'Grand_Total': grand
    })

yearly_totals = pd.DataFrame(rows)
yearly_totals.to_csv('data/yearly_totals.csv', index=False)
yearly_totals


In [ ]:
# 10. Bangun tabel breakdown Palsu per tahun emisi uang (hanya tersedia mulai 2020)
palsu_emisi = raw_parsed[
    (raw_parsed['MainCategory'] == 'Palsu') &
    (raw_parsed['SubLabel'].notna()) &
    (~raw_parsed['SubLabel'].astype(str).str.contains('Menunggu', na=False)) &
    (raw_parsed['Year'] >= 2020)
].copy()

palsu_emisi['TahunEmisi'] = palsu_emisi['SubLabel'].astype(str).str.replace('.0', '', regex=False)
palsu_by_tahun_emisi = palsu_emisi.groupby(['Year', 'TahunEmisi'], as_index=False)['Count'].sum()
palsu_by_tahun_emisi.to_csv('data/palsu_by_tahun_emisi.csv', index=False)
palsu_by_tahun_emisi


## 11. EDA Singkat
Beberapa visualisasi cepat untuk memahami pola data sebelum dibawa ke dashboard Streamlit.


In [ ]:
plt.figure(figsize=(9,5))
plt.plot(yearly_totals['Year'], yearly_totals['Grand_Total'], marker='o', linewidth=2, color='#c0392b')
plt.title('Total Temuan UPUR per Tahun - KPw BI Pematang Siantar')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Lembar/Keping')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.bar(yearly_totals['Year'], yearly_totals['Palsu'], color='#2c3e50')
plt.title('Jumlah Uang Palsu per Tahun')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Lembar/Keping')
plt.show()


In [ ]:
pivot = palsu_by_tahun_emisi.pivot(index='Year', columns='TahunEmisi', values='Count').fillna(0)
pivot.plot(kind='bar', stacked=True, figsize=(10,6), colormap='tab20')
plt.title('Komposisi Uang Palsu berdasarkan Tahun Emisi, per Tahun Temuan')
plt.xlabel('Tahun Temuan')
plt.ylabel('Jumlah Lembar/Keping')
plt.legend(title='Tahun Emisi', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()


## 12. Selesai
File `data/yearly_totals.csv` dan `data/palsu_by_tahun_emisi.csv` sudah siap.

**Langkah selanjutnya:**
1. Download kedua file CSV ini dari Colab (folder `data/`)
2. Push ke repo GitHub kamu di dalam folder `data/`
3. Pastikan `app.py` (Streamlit) ada di root repo, lalu deploy lewat [share.streamlit.io](https://share.streamlit.io)
